# Corpus and experimental populations

Trace the archived GoldAdjudicated populations, comment split and video source grouping.

**Scope:** These notebooks recompute descriptive statistics from the archived data and predictions. They do not retrain the original encoders or produce new evaluation populations.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score, average_precision_score, brier_score_loss

ROOT = Path.cwd().resolve()
if not (ROOT / "CITATION.cff").exists():
    ROOT = ROOT.parent
assert (ROOT / "CITATION.cff").exists(), "Run from the repository root or notebooks/"
print("Project:", ROOT.name)

## Recorded label counts
The following is a release population inventory, not an independent audit of the withheld source content.

In [ ]:
counts = pd.read_csv(ROOT / "data/labels/v1_1_GoldAdjudicated/POPULATION_COUNTS.csv")
display(counts)

## Source-disjoint video roles

In [ ]:
v = pd.read_csv(ROOT / "data/labels/v1_1_GoldAdjudicated/manifests/VIDEO_PRIMARY_MANIFEST.csv")
print("Aligned windows:", len(v), "Unique linked groups:", v.group_id.nunique())
print("Positive:", int(v.label_final.sum()), "Negative:", int((1-v.label_final).sum()))
assert len(v) == 3951 and v.group_id.nunique() == 214
assert (int(v.label_final.sum()),int((1-v.label_final).sum())) == (2326,1625)
display(v.groupby("platform").agg(windows=("master_id","size"),groups=("group_id","nunique")))

## Alignment exclusions

In [ ]:
display(pd.read_csv(ROOT / "results/statistics/EXCLUSION_CHARACTERISTICS.csv"))

## Comment split manifest
This selected subset is a different population from the video windows.

In [ ]:
c = pd.read_csv(ROOT / "data/manifests/comments/TEXT_SPLIT_MANIFEST.csv")
print("Comment manifest columns:", list(c.columns))
print("Rows:",len(c))
display(c.head(3))

## V4 revision: 5-role fold counts (`R1.1`) and 450 alignment exclusions (`R2.4`)
Inspect `ROLE_COUNTS.csv` and the aggregated exclusion breakdown in `EXCLUSION_REASON_PLATFORM_CLASS.csv` (`197` `frame_identity_match`, `200` `frame_identity_match+stt_content_match`, `45` `ocr_content_match`, `5` `frame_identity_match+ocr_content_match`, `3` all three failures; total `450` excluded, `3,951` retained).

In [ ]:
rc = pd.read_csv(ROOT / 'results/video/revision_20261108/roles/ROLE_COUNTS.csv')
display(rc.head(10))
ex = pd.read_csv(ROOT / 'results/statistics/revision_20261108/exclusions/EXCLUSION_REASON_PLATFORM_CLASS.csv')
display(ex.groupby(['excluded', 'reason_pattern'], as_index=False)['n'].sum())
assert int(ex.loc[ex.excluded, 'n'].sum()) == 450 and int(ex.loc[~ex.excluded, 'n'].sum()) == 3951